# 🎙️ ComfyUI + Fish Audio Drama 3 一鍵部署 (Google Colab)

本筆記本將在 Google Colab 上部署 **ComfyUI** 與 **[PedroMarinhoDev/ComfyUI-Drama3](https://github.com/PedroMarinhoDev/ComfyUI-Drama3)**（支援 35B MoE 情緒語音導演、多角色對話、零樣本聲音克隆與 Qwen3-ASR 語音轉錄）。

### ⚠️ 執行前必讀（硬體配置要求）
`Fish Audio Drama 3` 採用 **Experts-in-RAM** 架構（將龐大的 MoE 專家權重駐留於系統主記憶體，GPU 顯存僅需 ~6GB）。
- **推薦配置**：請點擊上方選單 **「執行階段 (Runtime)」 -> 「變更執行階段類型」**，選擇 **`L4 GPU`**（約 3.0 點/小時，內建 ~53GB 系統 RAM），或選擇 **`T4 GPU` 並開啟「高記憶體 (High-RAM)」**（約 1.67 點/小時，~51GB 系統 RAM）。
- **請勿使用標準版 T4**（僅 12.7GB 系統 RAM，載入模型時必定發生 OOM 當機）。
- **用畢省點提醒**：使用完畢後，請務必點擊右上角資源面板 **「中斷連線並刪除執行階段」** 以停止消耗計算單元。

In [ ]:
#@title 1. 🔍 檢查系統記憶體 (System RAM) 與 GPU 規格
import psutil
import shutil
import torch

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
disk_free_gb = shutil.disk_usage("/").free / (1024 ** 3)

print(f"🧠 系統主記憶體 (System RAM): {ram_gb:.1f} GB")
print(f"💾 可用磁碟空間 (Free Disk) : {disk_free_gb:.1f} GB")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"🎮 GPU 型號: {gpu_name} (VRAM: {vram_gb:.1f} GB)")
else:
    raise RuntimeError("❌ 未偵測到 GPU！請在 [執行階段] -> [變更執行階段類型] 選擇 L4 或 T4 (High-RAM) GPU。")

if ram_gb < 30:
    print("\n" + "=" * 70)
    print("❌ 【嚴重警告】目前系統記憶體不足 30 GB！")
    print("Drama 3 採用 Experts-in-RAM 架構，標準 T4 (12.7GB RAM) 載入模型必定崩潰。")
    print("👉 請立即前往 [執行階段] -> [變更執行階段類型]，切換至 L4 GPU 或開啟「高記憶體 (High-RAM)」！")
    print("=" * 70)
    raise SystemExit("請先切換為高記憶體 (High-RAM) 執行階段再重新執行。")
elif ram_gb < 45:
    print("\n⚠️ 系統記憶體介於 30~45 GB，建議在步驟 3 選擇 w4a8 或 w6a8 量化版本。")
else:
    print("\n✅ 硬體規格完美！系統記憶體充足，可流暢執行滿血版 int8 / fp8 模型。")

In [ ]:
#@title 2. 🛠️ 安裝 ComfyUI、ComfyUI-Manager 與 ComfyUI-Drama3
import os

COMFY_DIR = "/content/ComfyUI"

print("⏳ [1/3] 正在安裝系統工具 (aria2, ffmpeg) 與 ComfyUI 主程式...")
!apt-get update -qq && apt-get install -y -qq aria2 ffmpeg
if not os.path.exists(COMFY_DIR):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}

%cd {COMFY_DIR}
!pip install -q -r requirements.txt
!pip install -q "huggingface_hub[hf_transfer]"

print("⏳ [2/3] 正在安裝 ComfyUI-Manager 與 PedroMarinhoDev/ComfyUI-Drama3...")
custom_nodes_dir = os.path.join(COMFY_DIR, "custom_nodes")
os.makedirs(custom_nodes_dir, exist_ok=True)

manager_dir = os.path.join(custom_nodes_dir, "ComfyUI-Manager")
if not os.path.exists(manager_dir):
    !git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager.git {manager_dir}

drama3_dir = os.path.join(custom_nodes_dir, "ComfyUI-Drama3")
if not os.path.exists(drama3_dir):
    !git clone --depth 1 https://github.com/PedroMarinhoDev/ComfyUI-Drama3.git {drama3_dir}

print("✅ ComfyUI 與 ComfyUI-Drama3 節點安裝完成！（ComfyUI-Drama3 為原生節點，無需額外 pip 套件）")

In [ ]:
#@title 3. 📥 高速下載 Fish Audio Drama 3 模型與 Qwen3-ASR
#@markdown 請根據你的系統記憶體選擇主模型版本（預設推薦 `int8`，在消費級/L4 顯卡上比 `fp8` 快約 15%）：
model_variant = "drama3_int8experts_bf16.safetensors" #@param ["drama3_int8experts_bf16.safetensors", "drama3_w6a8experts_bf16.safetensors", "drama3_w4a8experts_bf16.safetensors", "drama3_fp8experts_bf16.safetensors"]
download_qwen3_asr = True #@param {type:"boolean"}

import os
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
REPO_ID = "PedroMarinhoDev/Drama-3-ComfyUI"
ASR_REPO_ID = "PedroMarinhoDev/Qwen3-ASR-ComfyUI"
COMFY_DIR = "/content/ComfyUI"

diff_dir = os.path.join(COMFY_DIR, "models", "diffusion_models")
vae_dir = os.path.join(COMFY_DIR, "models", "vae")
asr_dir = os.path.join(COMFY_DIR, "models", "asr")

os.makedirs(diff_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)
os.makedirs(asr_dir, exist_ok=True)

print("⏳ [1/3] 正在下載聲碼器 drama3_codec.safetensors (~1.6 GB) 至 models/vae/ ...")
hf_hub_download(
    repo_id=REPO_ID,
    filename="drama3_codec.safetensors",
    local_dir=vae_dir
)

if download_qwen3_asr:
    print("⏳ [2/3] 正在下載語音辨識模型 qwen3_asr_1.7b_bf16.safetensors (~4.1 GB) 至 models/asr/ ...")
    hf_hub_download(
        repo_id=ASR_REPO_ID,
        filename="qwen3_asr_1.7b_bf16.safetensors",
        local_dir=asr_dir
    )
else:
    print("⏭️ [2/3] 已略過下載 Qwen3-ASR 模型。")

print(f"⏳ [3/3] 正在高速下載主模型 {model_variant} 至 models/diffusion_models/ (請稍候約 1~2 分鐘)...")
hf_hub_download(
    repo_id=REPO_ID,
    filename=model_variant,
    local_dir=diff_dir
)

print("🎉 所有模型下載完畢！可以執行下一步啟動 ComfyUI 網頁介面。")

In [ ]:
#@title 4. 🚀 啟動 Cloudflare Tunnel 與 ComfyUI 網頁介面
#@markdown 執行後請等待約 15~30 秒，點擊下方印出的 `https://xxxx.trycloudflare.com` 連結即可開啟 ComfyUI 介面。
import subprocess
import threading
import time
import socket
import re
import os

if not os.path.exists("/usr/bin/cloudflared"):
    print("⏳ 正在安裝 Cloudflare Tunnel (cloudflared)...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

def start_cloudflared(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break
    print("\n" + "=" * 70)
    print("🌐 ComfyUI 本地伺服器已就緒，正在建立 Cloudflare 安全通道...")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    for line in p.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print(f"👉 點擊此網址開啟 ComfyUI: {match.group(0)}")
            print("=" * 70 + "\n")

threading.Thread(target=start_cloudflared, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
!python main.py --dont-print-server